In [4]:
import glob
import numpy as np
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.bam import MultiBAMv4

import importlib
import utils.my_lora_utils
importlib.reload(utils.my_lora_utils)
from utils.my_lora_utils import *

print(utils.my_lora_utils.__file__)

def parse_gt_from_filename(f):
    parts = f.split("_")
    return parts[6]

c:\Users\priba\Sean-2025\INC-LAB\BAM\INC-BAM\utils\my_lora_utils.py


In [5]:
######## Parameters can be adjusted ######
sf = 9      
N = 2**sf
input_row = 512
input_col = 33

second_layer = 1024
final_layer = 8

eta=1e-5
num_epochs=10
batch_size=16
######## Parameters can be adjusted ######

In [6]:
dataset_name_folder = f"dataset_sf{sf}_{input_row}x{input_col}"
dataset_clean_name_folder = f"clean_dataset_sf{sf}_{input_row}x{input_col}"
input_layer = input_row * input_col
layers = [input_layer, second_layer , final_layer]  # compress 3840 → 1024 → 256

GEENRATE_ = True #### Secure accidently running

################### Load all .npy files ########################################
print("LOAD DATASET")
files = glob.glob(f'{dataset_name_folder}/*.npy')
data_list = []
gt_list = []
database_clean_signal = []
for f in files:
    x = np.load(f)
    gt_symbol = parse_gt_from_filename(f)
    data_list.append(x.flatten())
    gt_list.append(int(gt_symbol)) 

for sym in range(N): # 0 until 2**sf
    file_str = f'{dataset_clean_name_folder}/s_sf{sf}_bw125_{sym}.npy'
    x = np.load(file_str)
    x_flat= x.flatten()
    database_clean_signal.append(x_flat)
    
X = np.array(data_list)
gt_list = np.array(gt_list)
database_clean_signal = np.array(database_clean_signal)

################### Load all .npy files ########################################

multi_bam = MultiBAMv4(layers_dims=layers, eta=eta)

if (GEENRATE_):
    
    folder_path = f"weight_{input_layer}_{second_layer}_{final_layer}"

    # Check if folder exists, if not create it
    check_and_make_folder(folder_path)
    layer_losses = multi_bam.train(X=X, Y_sym = gt_list, database=database_clean_signal, num_epochs=num_epochs, batch_size=batch_size)
    
    for i, bam in enumerate(multi_bam.bams):
        np.save(f"{folder_path}/weights_layer_{i}.npy", bam.W)

def load_weight(): 
    ## HOW TO LOAD WEIGHT
    layers = [input_row * input_col, second_layer, final_layer] # <-- must match training

    multi_bam = MultiBAMv4(layers_dims=layers, eta=eta)
    for i, bam in enumerate(multi_bam.bams):
        bam.W = np.load(f"weight/weights_layer_{i}.npy")


LOAD DATASET
Folder created: weight_16896_1024_8

--- Training Layer 1/2 ---
Epoch 1/10, MSE=0.017514
Epoch 2/10, MSE=0.011453
Epoch 3/10, MSE=0.011073
Epoch 4/10, MSE=0.010928
Epoch 5/10, MSE=0.010850
Epoch 6/10, MSE=0.010804
Epoch 7/10, MSE=0.010770
Epoch 8/10, MSE=0.010750
Epoch 9/10, MSE=0.010733
Epoch 10/10, MSE=0.010719

--- Training Layer 2/2 ---
Epoch 1/10, MSE=0.275627
Epoch 2/10, MSE=0.169902
Epoch 3/10, MSE=0.169649
Epoch 4/10, MSE=0.169631
Epoch 5/10, MSE=0.169644
Epoch 6/10, MSE=0.169625
Epoch 7/10, MSE=0.169634
Epoch 8/10, MSE=0.169578
Epoch 9/10, MSE=0.169578
Epoch 10/10, MSE=0.169567
